# Results analysis

**Purpose**: Aggregate experiment outputs and produce plots + statistical tests to compare strategies.

## Prerequisites
- Result JSON files in `../results/` (update `results_dir` if needed).
- scipy and statsmodels installed for ANOVA/Tukey tests.

## How to run
- Run top-to-bottom; figures and reports are saved into `../results/analysis/`.

## Notes
- Paths in this notebook are repository- and machine-specific; update them before running.
- If you're running from the `notebooks/` folder, the `PROJECT_ROOT` logic should work as-is.


## Result Analysis Notebook
Parse multiple JSON result files from `../results/` and generate summary tables, plots, and basic statistical comparisons.

**Expected input**: each JSON file contains at least `final_ap` (or `history.val_ap`).

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import yaml
from pathlib import Path
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Set plotting style

In [ ]:
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("tab10")

## Load results from files

## Loading results
Each file in `results_dir` is treated as a separate experiment run.
If your schema differs, adjust `extract_metrics()` accordingly.

In [ ]:
results_dir = Path("../results")
result_files = list(results_dir.glob("*.json"))

print(f"Found {len(result_files)} result files")

# Load and parse results
all_results = {}
for file_path in result_files:
    try:
        with open(file_path, 'r') as f:
            result = json.load(f)
        exp_name = file_path.stem
        all_results[exp_name] = result
        print(f"✓ Loaded: {exp_name}")
    except Exception as e:
        print(f"✗ Error loading {file_path}: {e}")

# Create analysis directory
analysis_dir = Path("../results/analysis")
analysis_dir.mkdir(parents=True, exist_ok=True)

## Function to extract metrics from results

In [ ]:
def extract_metrics(results_dict):
    """Extract key metrics from results dictionary."""
    metrics_list = []
    
    for exp_name, result in results_dict.items():
        metrics = {
            'experiment': exp_name,
            'strategy': exp_name.split('_')[0] if '_' in exp_name else exp_name
        }
        
        # Extract final AP
        if 'final_ap' in result:
            metrics['final_ap'] = result['final_ap']
        elif 'history' in result and 'val_ap' in result['history']:
            val_ap = result['history']['val_ap']
            metrics['final_ap'] = val_ap[-1] if val_ap else 0
        
        # Extract best AP
        if 'best_ap' in result:
            metrics['best_ap'] = result['best_ap']
        
        # Extract labeled counts
        if 'final_labeled_count' in result:
            metrics['final_labeled'] = result['final_labeled_count']
        
        # Extract timing if available
        if 'timing' in result:
            timing = result['timing']
            if isinstance(timing, dict):
                metrics.update({f"time_{k}": v for k, v in timing.items()})
        
        metrics_list.append(metrics)
    
    return pd.DataFrame(metrics_list)

## Create metrics DataFrame

In [ ]:
df_metrics = extract_metrics(all_results)

print("\nMetrics Summary:")
print(f"Total experiments: {len(df_metrics)}")
print(f"Unique strategies: {df_metrics['strategy'].nunique()}")

# Display metrics
if len(df_metrics) > 0:
    print("\nTop performing experiments:")
    top_experiments = df_metrics.sort_values('final_ap', ascending=False).head(10)
    print(top_experiments[['experiment', 'strategy', 'final_ap', 'best_ap']].to_string())

## Plot 1: Performance distribution

In [ ]:
plt.figure(figsize=(12, 6))

# Group by strategy
if 'strategy' in df_metrics.columns and 'final_ap' in df_metrics.columns:
    grouped = df_metrics.groupby('strategy')['final_ap']
    
    # Create box plot
    strategies = sorted(df_metrics['strategy'].unique())
    data = [df_metrics[df_metrics['strategy'] == s]['final_ap'].values for s in strategies]
    
    plt.boxplot(data, labels=strategies)
    plt.xticks(rotation=45, ha='right')
    plt.ylabel('Final AP')
    plt.title('Performance Distribution by Strategy')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(analysis_dir / "performance_distribution.png", dpi=300, bbox_inches='tight')
    plt.show()

## Plot 2: Correlation analysis

In [ ]:
if len(df_metrics) > 1:
    # Select numeric columns for correlation
    numeric_cols = df_metrics.select_dtypes(include=[np.number]).columns
    
    if len(numeric_cols) > 1:
        # Calculate correlation matrix
        corr_matrix = df_metrics[numeric_cols].corr()
        
        plt.figure(figsize=(10, 8))
        sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0,
                   square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
        plt.title('Correlation Matrix of Metrics')
        plt.tight_layout()
        plt.savefig(analysis_dir / "correlation_matrix.png", dpi=300, bbox_inches='tight')
        plt.show()

## Statistical analysis

In [ ]:
print("\n" + "="*60)
print("STATISTICAL ANALYSIS")
print("="*60)

if 'strategy' in df_metrics.columns and 'final_ap' in df_metrics.columns:
    # Group by strategy for statistical tests
    strategy_groups = {}
    for strategy in df_metrics['strategy'].unique():
        ap_values = df_metrics[df_metrics['strategy'] == strategy]['final_ap'].values
        if len(ap_values) > 1:
            strategy_groups[strategy] = ap_values
    
    # Perform ANOVA if we have multiple groups
    if len(strategy_groups) >= 2:
        anova_data = list(strategy_groups.values())
        
        # Check if we have enough data
        if all(len(g) > 1 for g in anova_data):
            f_stat, p_value = stats.f_oneway(*anova_data)
            
            print(f"One-way ANOVA Results:")
            print(f"F-statistic: {f_stat:.4f}")
            print(f"P-value: {p_value:.4f}")
            
            if p_value < 0.05:
                print("Significant difference between strategies (p < 0.05)")
                
                # Post-hoc tests
                print("\nPost-hoc Tukey HSD Test:")
                
                # Prepare data for Tukey test
                tukey_data = []
                tukey_labels = []
                for strategy, values in strategy_groups.items():
                    tukey_data.extend(values)
                    tukey_labels.extend([strategy] * len(values))
                
                # Perform Tukey test
                from statsmodels.stats.multicomp import pairwise_tukeyhsd
                
                tukey_result = pairwise_tukeyhsd(tukey_data, tukey_labels, alpha=0.05)
                print(tukey_result)
                
                # Plot Tukey results
                fig = tukey_result.plot_simultaneous()
                plt.title('Tukey HSD Test Results')
                plt.tight_layout()
                plt.savefig(analysis_dir / "tukey_test.png", dpi=300, bbox_inches='tight')
                plt.show()
            else:
                print("No significant difference between strategies")
        else:
            print("Insufficient data for ANOVA test")
    
    # Pairwise t-tests for top strategies
    if len(strategy_groups) >= 2:
        print("\nPairwise T-tests (top 3 strategies):")
        
        # Get top 3 strategies by mean AP
        strategy_means = {s: np.mean(vals) for s, vals in strategy_groups.items()}
        top_strategies = sorted(strategy_means.items(), key=lambda x: x[1], reverse=True)[:3]
        
        for i in range(len(top_strategies)):
            for j in range(i + 1, len(top_strategies)):
                strat1, mean1 = top_strategies[i]
                strat2, mean2 = top_strategies[j]
                
                if strat1 in strategy_groups and strat2 in strategy_groups:
                    t_stat, p_val = stats.ttest_ind(
                        strategy_groups[strat1],
                        strategy_groups[strat2],
                        equal_var=False
                    )
                    
                    print(f"{strat1} vs {strat2}:")
                    print(f"  t-statistic: {t_stat:.4f}, p-value: {p_val:.4f}")
                    if p_val < 0.05:
                        print(f"  Significant difference (p < 0.05)")
                    print()

## Plot 3: Learning trajectory analysis

In [ ]:
print("\n" + "="*60)
print("LEARNING TRAJECTORY ANALYSIS")
print("="*60)

# Analyze learning curves if available
for exp_name, result in list(all_results.items())[:5]:  # Analyze first 5 experiments
    if 'history' in result and 'val_ap' in result['history']:
        val_ap = result['history']['val_ap']
        
        if len(val_ap) > 1:
            # Calculate learning rate (AP gain per epoch)
            ap_gains = np.diff(val_ap)
            avg_gain = np.mean(ap_gains) if len(ap_gains) > 0 else 0
            
            # Calculate convergence metrics
            if len(val_ap) >= 3:
                # Slope of last 3 points
                x = np.arange(len(val_ap) - 3, len(val_ap))
                y = val_ap[-3:]
                slope, intercept = np.polyfit(x, y, 1)
                
                print(f"{exp_name}:")
                print(f"  Final AP: {val_ap[-1]:.4f}")
                print(f"  Average AP gain per epoch: {avg_gain:.4f}")
                print(f"  Final slope: {slope:.4f} (negative = converging)")
                print(f"  Total improvement: {val_ap[-1] - val_ap[0]:.4f}")
                print()

## Plot 4: Query efficiency analysis

In [ ]:
print("\n" + "="*60)
print("QUERY EFFICIENCY ANALYSIS")
print("="*60)

# Analyze query patterns
query_efficiency_data = []
for exp_name, result in all_results.items():
    if 'history' in result and 'val_ap' in result['history']:
        val_ap = result['history']['val_ap']
        
        if len(val_ap) > 1:
            total_gain = val_ap[-1] - val_ap[0]
            
            # Estimate number of queries (simplified)
            # In practice, you'd track actual queries
            strategy = exp_name.split('_')[0] if '_' in exp_name else exp_name
            
            query_efficiency_data.append({
                'experiment': exp_name,
                'strategy': strategy,
                'initial_ap': val_ap[0],
                'final_ap': val_ap[-1],
                'total_gain': total_gain,
                'relative_gain': total_gain / max(val_ap[0], 0.001)  # Avoid division by zero
            })

if query_efficiency_data:
    df_efficiency = pd.DataFrame(query_efficiency_data)
    
    # Plot query efficiency
    plt.figure(figsize=(12, 6))
    
    for strategy in df_efficiency['strategy'].unique():
        strat_data = df_efficiency[df_efficiency['strategy'] == strategy]
        plt.scatter(strat_data['initial_ap'], strat_data['total_gain'], 
                   label=strategy, s=100, alpha=0.7)
    
    plt.xlabel('Initial AP')
    plt.ylabel('Total AP Gain')
    plt.title('Query Efficiency: Gain vs Initial Performance')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(analysis_dir / "query_efficiency.png", dpi=300, bbox_inches='tight')
    plt.show()
    
    # Display efficiency metrics
    print("\nQuery Efficiency Summary:")
    efficiency_summary = df_efficiency.groupby('strategy').agg({
        'total_gain': ['mean', 'std', 'count'],
        'relative_gain': ['mean', 'std']
    }).round(4)
    
    print(efficiency_summary)

## Generate comprehensive report

In [ ]:
print("\n" + "="*60)
print("COMPREHENSIVE ANALYSIS REPORT")
print("="*60)

report_lines = []

# 1. Executive Summary
report_lines.append("EXECUTIVE SUMMARY")
report_lines.append("-" * 40)

if len(df_metrics) > 0:
    best_exp = df_metrics.loc[df_metrics['final_ap'].idxmax()]
    worst_exp = df_metrics.loc[df_metrics['final_ap'].idxmin()]
    
    report_lines.append(f"Total Experiments: {len(df_metrics)}")
    report_lines.append(f"Best Performance: {best_exp['experiment']} ({best_exp['final_ap']:.3f})")
    report_lines.append(f"Worst Performance: {worst_exp['experiment']} ({worst_exp['final_ap']:.3f})")
    report_lines.append(f"Average Performance: {df_metrics['final_ap'].mean():.3f} ± {df_metrics['final_ap'].std():.3f}")

# 2. Strategy Rankings
report_lines.append("\nSTRATEGY RANKINGS")
report_lines.append("-" * 40)

if 'strategy' in df_metrics.columns:
    strategy_stats = df_metrics.groupby('strategy')['final_ap'].agg(['mean', 'std', 'count', 'min', 'max'])
    strategy_stats = strategy_stats.sort_values('mean', ascending=False)
    
    for i, (strategy, stats) in enumerate(strategy_stats.iterrows(), 1):
        report_lines.append(f"{i}. {strategy}:")
        report_lines.append(f"   Mean AP: {stats['mean']:.3f} ± {stats['std']:.3f}")
        report_lines.append(f"   Range: {stats['min']:.3f} - {stats['max']:.3f}")
        report_lines.append(f"   Samples: {int(stats['count'])}")

# 3. Key Findings
report_lines.append("\nKEY FINDINGS")
report_lines.append("-" * 40)

if len(df_metrics) > 0:
    # Calculate effect sizes
    if 'strategy' in df_metrics.columns:
        top_strategy = strategy_stats.index[0]
        bottom_strategy = strategy_stats.index[-1]
        
        top_mean = strategy_stats.loc[top_strategy, 'mean']
        bottom_mean = strategy_stats.loc[bottom_strategy, 'mean']
        
        effect_size = (top_mean - bottom_mean) / df_metrics['final_ap'].std()
        
        report_lines.append(f"1. Performance Range: {df_metrics['final_ap'].min():.3f} - {df_metrics['final_ap'].max():.3f}")
        report_lines.append(f"2. Best Strategy: {top_strategy} ({top_mean:.3f})")
        report_lines.append(f"3. Worst Strategy: {bottom_strategy} ({bottom_mean:.3f})")
        report_lines.append(f"4. Effect Size (Cohen's d): {effect_size:.2f}")
        
        if effect_size > 0.8:
            report_lines.append("   (Large practical significance)")
        elif effect_size > 0.5:
            report_lines.append("   (Medium practical significance)")
        elif effect_size > 0.2:
            report_lines.append("   (Small practical significance)")
        else:
            report_lines.append("   (Negligible practical significance)")

# 4. Recommendations
report_lines.append("\nRECOMMENDATIONS")
report_lines.append("-" * 40)

if len(df_metrics) > 0:
    report_lines.append("1. Primary Recommendation:")
    report_lines.append(f"   Use {top_strategy} for best overall performance")
    
    report_lines.append("\n2. Secondary Recommendations:")
    report_lines.append("   - For computational efficiency: Consider simpler strategies")
    report_lines.append("   - For data efficiency: Focus on uncertainty-based methods")
    report_lines.append("   - For robustness: Use ensemble or hybrid approaches")
    
    report_lines.append("\n3. Future Work:")
    report_lines.append("   - Test on larger datasets")
    report_lines.append("   - Explore strategy combinations")
    report_lines.append("   - Investigate domain adaptation effects")

# Save report
report_path = analysis_dir / "analysis_report.txt"
with open(report_path, 'w') as f:
    f.write("\n".join(report_lines))

print(f"\nAnalysis report saved to: {report_path}")

# Display report
print("\n".join(report_lines[:50]))  # Show first 50 lines

print(f"\nAnalysis complete! Results saved in: {analysis_dir}")
print("Files created:")
for file in analysis_dir.glob("*"):
    print(f"  - {file.name}")